In [3]:
# ============================================================================
# Trigger-time preselection plot (Howard Fig. 7/8 style), from the
# `trigger_time` tree in selection/toml/preselection.toml.
#
# `trigger_time` is a mode = "event" tree: one row per triggered event, not
# one row per reconstructed interaction like `preselection`. That is a real
# structural consequence of how selection/src/framework.cc dispatches branch
# construction (see the comment block in preselection.toml) -- not a
# simplification chosen here. A mode = "event" tree can only carry branches
# of type = "event" (framework.cc's Mode::Event block only recognizes that
# one type), so `true_pdg` / `true_neutrino_energy` (both type = "mctruth")
# are NOT available directly on this tree -- there is no way to compute the
# PPFX weight from `trigger_time` alone.
#
# PPFX weight, via a join to `preselection`:
#   Both `trigger_time` and `preselection` carry Run/Subrun/Evt columns, so
#   the per-event PPFX weight is obtained by building a
#   (Run, Subrun, Evt) -> (true_pdg, true_neutrino_energy) lookup from
#   `events/nominal/preselection` and merging it onto `trigger_time`. This
#   join is NOT complete: `preselection`'s cuts include `fiducial_cut` (a
#   reco-level, row-filtering cut -- only interactions reconstructed inside
#   the fiducial volume get a row there), while `trigger_time`'s only cuts
#   are the two event-level ones. So a `trigger_time` row for an event with
#   no fiducial-volume interaction (none reconstructed, or reconstructed
#   outside the FV) will not find a match. The match fraction is measured
#   and printed below rather than assumed; unmatched rows get PPFX weight
#   1.0 (the same fallback compute_ppfx already uses for non-numu/numubar
#   PDGs), which is reported explicitly so it isn't a silent bias.
#
# Rock muon treatment -- Roy's approach (NuMu_CC_Inclusive_Technote,
# Sec. 4.3/4.4): the rock sample is a separate MC production, scaled
# directly to the data exposure by POT ratio (same as our nominal MC/rock
# scaling), and shown as its own component in the trigger-time comparison
# (her Fig. 13b) without any truth-matching-based split. Background
# suppression is handled elsewhere in her selection chain via geometric/
# topological rejection cuts (fiducial-volume vertex requirement, rejecting
# tracks that enter from the detector boundary, CRT timing), not by
# filtering the rock sample itself with a truth-matching criterion. We
# follow the same approach here: the "rock included" case below uses the
# full rock/trigger_time sample restricted to fiducial-volume interactions
# (the FV requirement is the geometric cut, analogous to Roy's), with no
# additional true_category/n_true_nu-based filter on top. (An earlier
# version of this cell tried isolating "genuine" rock muons via
# true_category != 9, and a finer n_true_nu-based criterion on top of that;
# both were dropped in favor of matching Roy's simpler, cut-based treatment.)
#
# Rock is NOT PPFX-reweighted (standing instruction: rock muons are not beam
# neutrinos).
#
# Self-contained: re-imports and re-defines everything it needs.
# ============================================================================

import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import uproot

print("CELL VERSION: trigger_time_v22_trimmed_plots_2026-08-20")

# ------------------------------------------------------------------ config --
DATA_FILE = ("/Users/rvizarreta/Library/CloudStorage/GoogleDrive-rvizarreta14@gmail.com/"
             "My Drive/\U0001f3db PhD Repository/\U0001f680 Research/"
             "\U0001f916 Experiments&Projects/ICARUS/ICARUS_CC0pi_Selection/"
             "data/preselection.root")

FLUX_FILE = ("/Users/rvizarreta/Library/CloudStorage/GoogleDrive-rvizarreta14@gmail.com/"
             "My Drive/\U0001f3db PhD Repository/\U0001f680 Research/"
             "\U0001f916 Experiments&Projects/ICARUS/ICARUS_CC0pi_Selection/"
             "systematics/2025-04-08_out_450.37_7991.98_79512.66.root")

OUTPUT_DIR = ("/Users/rvizarreta/Library/CloudStorage/GoogleDrive-rvizarreta14@gmail.com/"
              "My Drive/\U0001f3db PhD Repository/\U0001f680 Research/"
              "\U0001f916 Experiments&Projects/ICARUS/ICARUS_CC0pi_Selection/"
              "ICARUS-NuMI-CC0pi-Selection/spineplot/myAnalysis/"
              "1muNp0pi_Nge1_uncontained/preselection")

ONBEAM_POT  = 2.36124e19
MC_SAMPLE   = "nominal"
TREE        = "trigger_time"
horn_current = 'fhc'

# ------------------------------------------------------------------ style ---
def apply_tick_style(ax):
    ax.tick_params(axis='both', which='major', labelsize=10, size=8, width=2, direction='in')
    ax.minorticks_on()
    ax.tick_params(axis='both', which='minor', size=4, width=1, direction='in')
    for lbl in ax.get_xticklabels() + ax.get_yticklabels():
        lbl.set_fontfamily('sans-serif')
    ax.yaxis.get_offset_text().set_fontfamily('sans-serif')
    ax.xaxis.get_offset_text().set_fontfamily('sans-serif')

def add_icarus_label(ax, subtitle="Work-in-Progress"):
    yr, xr = ax.get_ylim(), ax.get_xlim()
    ax.text(xr[0] + 0.01 * (xr[1] - xr[0]), yr[1] + 0.02 * (yr[1] - yr[0]),
            r'$\bf{ICARUS \cdot NuMI}$ Data (10% Run 2)' + '\n' + subtitle,
            fontsize=8, color="chocolate", verticalalignment="bottom",
            fontfamily="sans-serif")

def add_pot_label(ax, pot_str):
    yr, xr = ax.get_ylim(), ax.get_xlim()
    ax.text(xr[1] - 0.01 * (xr[1] - xr[0]), yr[1] + 0.02 * (yr[1] - yr[0]), pot_str,
            fontsize=8, color="black", verticalalignment="bottom",
            horizontalalignment="right", fontfamily="sans-serif")

def add_corner_label(ax, text, y=0.85):
    ax.text(0.97, y, text, transform=ax.transAxes, fontsize=7, color='black',
            horizontalalignment='right', verticalalignment='top',
            fontfamily="sans-serif")

# ------------------------------------------------------------------- load ---
f = uproot.open(DATA_FILE)
flux = uproot.open(FLUX_FILE)
ppfx_numu    = flux[f'ppfx_flux_weights/hweights_{horn_current}_numu;1']
ppfx_numubar = flux[f'ppfx_flux_weights/hweights_{horn_current}_numubar;1']
numu_values, numu_edges       = ppfx_numu.values(),    ppfx_numu.axes[0].edges()
numubar_values, numubar_edges = ppfx_numubar.values(), ppfx_numubar.axes[0].edges()


def read_exposure(sample, tree=TREE):
    """POT/Livetime histograms if present, else the <tree>_exposure tree.
    Prints which source it used and the raw value(s), every time -- not
    just on a naming mismatch -- so this is auditable rather than a black
    box for a sample like rock where the number matters a lot."""
    pot = lt = None
    if f"events/{sample}/POT" in f:
        pot = f[f"events/{sample}/POT"].values().sum()
        print(f"    {sample}: POT read from events/{sample}/POT histogram = {pot:.6e}")
    if f"events/{sample}/Livetime" in f:
        lt = f[f"events/{sample}/Livetime"].values().sum()
        print(f"    {sample}: Livetime read from events/{sample}/Livetime histogram = {lt:.6e}")
    if pot is not None or lt is not None:
        return pot, lt

    key = f"events/{sample}/{tree}_exposure"
    if key not in f:
        raise KeyError(f"no exposure information found for sample '{sample}' "
                       f"(looked for events/{sample}/POT, /Livetime and {key})")
    t = f[key]
    print(f"    {sample}/{tree}_exposure branches: {t.keys()}")
    arrays = t.arrays(library="np")
    for name, vals in arrays.items():
        low = name.lower()
        if pot is None and "pot" in low:
            pot = float(np.nansum(vals))
            print(f"    {sample}: POT summed from {tree}_exposure branch '{name}' = {pot:.6e}")
        if lt is None and ("livetime" in low or "live_time" in low):
            lt = float(np.nansum(vals))
            print(f"    {sample}: Livetime summed from {tree}_exposure branch '{name}' = {lt:.6e}")
    return pot, lt


print("Reading exposures (trigger_time tree)...")
onbeam_pot_raw, onbeam_lt  = read_exposure("onbeam")
_,              offbeam_lt_raw = read_exposure("offbeam")
rock_pot,        _      = read_exposure("rock")
nominal_pot_raw, _      = read_exposure(MC_SAMPLE)

for label, val in (("onbeam POT (raw)", onbeam_pot_raw), ("onbeam livetime", onbeam_lt),
                   ("offbeam livetime (raw)", offbeam_lt_raw),
                   ("rock POT", rock_pot), ("nominal MC POT (raw)", nominal_pot_raw)):
    if val is None or not np.isfinite(val) or val <= 0:
        raise ValueError(f"{label} came back as {val}. Check the exposure branch "
                         f"names printed above.")

# Offbeam livetime correction -- REMOVED (was v14/v15's
# OFFBEAM_LIVETIME_CORRECTION_FACTOR = ONBEAM_POT / onbeam_pot_raw, applied
# as a proportional correction to offbeam_lt). That factor was derived from
# the Moreno NuMI POT-counting closure test, whose finding was specifically
# that CAF metadata (sr->hdr.pot) is only stored on the FIRST EVENT of each
# 50-event CAF file, so any per-event SUM OF POT undercounts for a
# prescaled/reduced sample. Livetime here is not a per-event sum of a
# per-event-stored quantity -- it's a count of gates/spills (onbeam Livetime
# = 483,958, close to the 473,175 "good spills" figure from the closure
# test) -- so the specific undercounting mechanism identified for POT
# summing has no clear analog for a gate count, and applying that
# POT-derived factor to livetime was not actually justified. Using the raw
# livetime values directly instead.
offbeam_lt = offbeam_lt_raw
print(f"\nOnbeam POT (raw, from file)                : {onbeam_pot_raw:.6e}")
print(f"Onbeam POT (validated, histogram-integration): {ONBEAM_POT:.6e}  "
      f"(Moreno, NuMI POT Counting Closure Test, Apr 1 2025 -- used for ONBEAM_POT/mc_scale/"
      f"rock_scale only; NOT applied to offbeam livetime, see comment above)")
print(f"Offbeam livetime (raw, from file, used)     : {offbeam_lt:.6e}")

# preselection.root was rebuilt (2026-08-19) with a much larger job set.
# nominal_pot_raw, read directly from the file's own events/nominal/POT
# histogram above, matches (to 5 sig figs) the raw COMPLETE-production POT
# value (7.78520e+20) that the broken/recovered-hadd-file correction was
# derived from -- so that correction applies directly:
# raw 7.78520e+20 -> corrected (true, recovered-file-loss-accounted) POT
# 7.178390e+20, i.e. POT_CORRECTION_FACTOR = 7.178390e20 / 7.78520e20.
POT_CORRECTION_FACTOR = 7.178390e20 / 7.78520e20
nominal_pot = nominal_pot_raw * POT_CORRECTION_FACTOR

mc_scale      = ONBEAM_POT / nominal_pot
offbeam_scale = onbeam_lt / offbeam_lt
rock_scale    = ONBEAM_POT / rock_pot

print(f"Nominal MC POT (raw, from file)      : {nominal_pot_raw:.6e}")
print(f"POT correction factor                : {POT_CORRECTION_FACTOR:.6f}  "
      f"(= 7.178390e+20 / 7.78520e+20, broken/recovered-hadd-file correction)")
print(f"Nominal MC POT (corrected, used)     : {nominal_pot:.6e}")
print(f"Rock POT (raw, from file)       : {rock_pot:.4e}")
print(f"MC scale:      {mc_scale:.4e}")
print(f"Offbeam scale: {offbeam_scale:.4e}")
print(f"Rock scale:    {rock_scale:.4e}")
print(f"\nRock scale / MC scale = {rock_scale/mc_scale:.2f}x -- i.e. each valid rock row "
      f"is weighted {rock_scale/mc_scale:.1f}x more than each valid nominal-MC row. "
      f"This ratio is exactly (nominal_pot / rock_pot) = "
      f"({nominal_pot:.4e} / {rock_pot:.4e}) = {nominal_pot/rock_pot:.2f}.")

print(f"\nLoading '{TREE}' trees...")
mc      = f[f"events/{MC_SAMPLE}/{TREE}"].arrays(library="pd")
rock    = f[f"events/rock/{TREE}"].arrays(library="pd")
onbeam  = f[f"events/onbeam/{TREE}"].arrays(library="pd")
offbeam = f[f"events/offbeam/{TREE}"].arrays(library="pd")

for name, df in ((MC_SAMPLE, mc), ("rock", rock), ("onbeam", onbeam), ("offbeam", offbeam)):
    print(f"  {name:>8}: {len(df)} rows, columns = {list(df.columns)}")

# ----------------------------------------------------- resolve branch name --
candidates = [c for c in mc.columns if "trigger_within_gate" in c]
if len(candidates) != 1:
    raise KeyError(f"expected exactly one column matching 'trigger_within_gate' "
                   f"in the {TREE} tree, found {candidates}. "
                   f"Available columns: {list(mc.columns)}")
TRIGGER_BRANCH = candidates[0]
print(f"\nUsing branch '{TRIGGER_BRANCH}' for trigger time.")

for name, df in ((MC_SAMPLE, mc), ("rock", rock), ("onbeam", onbeam), ("offbeam", offbeam)):
    missing = [k for k in ("Run", "Subrun", "Evt") if k not in df.columns]
    if missing:
        raise KeyError(f"{name}/{TREE} is missing join key column(s) {missing}")

# ------------------ rock: restrict to FV + angularly rock-like interactions
# `trigger_time` is a mode="event" tree with NO fiducial_cut -- event-mode
# trees cannot evaluate reco-level cuts (see the header comment). This
# restriction plays the role of Roy's rock-suppression criteria, applied at
# the (Run, Subrun, Evt) level via a join to rock/preselection:
#   1. FV: rock/preselection's own fiducial_cut has already confirmed a
#      genuine FV interaction exists for that event.
#   2. Angular: that interaction's longest_track direction must be
#      rock-like, not cosmic-like -- Cos(theta_LTYD) = start_dir_y (of the
#      normalized direction) above the numeric valley found in the
#      "rock-vs-cosmic angular separation" cell of this notebook
#      (2026-08-20 run: valley = 0.060, splitting a clean bimodal
#      distribution -- cosmic-like lobe peaked near -0.85, rock-like lobe
#      peaked near +0.85). Two independent truth-level cross-checks
#      (n_true_nu, true_category) were tried and both found uninformative
#      at the per-interaction granularity this needs (see that cell's
#      notes) -- this cut is justified by the shape itself plus the
#      physical picture (rock enters near-horizontal along the beam
#      direction, cosmics enter near-vertical), the same kind of argument
#      Roy's own Cos(theta_LTYD) > -0.7 cut relies on, not by a truth flag.
#      An event is kept if ANY of its FV interactions clears this bar --
#      it does not require every interaction in the event to.
ROCK_ANGULAR_VALLEY = 0.060

rock_presel = f["events/rock/preselection"].arrays(
    ["Run", "Subrun", "Evt", "reco_longest_track_length",
     "reco_longest_track_start_dir_x", "reco_longest_track_start_dir_y",
     "reco_longest_track_start_dir_z"], library="pd")
_vx = rock_presel["reco_longest_track_start_dir_x"].values
_vy = rock_presel["reco_longest_track_start_dir_y"].values
_vz = rock_presel["reco_longest_track_start_dir_z"].values
_norm = np.sqrt(_vx**2 + _vy**2 + _vz**2)
_valid_dir = np.isfinite(_norm) & (_norm > 0)
rock_presel["_cos_theta_y"] = np.nan
rock_presel.loc[_valid_dir, "_cos_theta_y"] = _vy[_valid_dir] / _norm[_valid_dir]

rock_fv_keyset = set(map(tuple, rock_presel[["Run", "Subrun", "Evt"]].values))

# IMPORTANT: each rock-triggering event has, on average, ~14.5 reconstructed
# interactions in preselection (SPINE over-splits rock events into many
# spurious candidates -- see the FV-restriction print below, which shows
# essentially all of them pass fiducial_cut). Deciding "rock-like" via an
# OR across all of an event's interactions (keep if ANY interaction passes
# the angular cut) is close to a no-op for the same reason FV alone was:
# with ~14.5 chances and a 33.5% per-interaction pass rate, the odds that
# NONE pass by chance are only ~(1-0.335)^14.5 ~ 0.3%, so almost every
# event "passes" regardless of its actual character. Instead, pick the
# SINGLE interaction per event with the longest longest_track (the one
# most likely to be the actual triggering particle) and base the decision
# on that one interaction's angle only.
#
# groupby(...).idxmax() raises if a group's "reco_longest_track_length" is
# all-NaN (no valid longest_track anywhere in that event) -- restrict to
# rows with a usable direction AND a usable length BEFORE the groupby, so
# idxmax never sees an all-NaN group. Events with no such row at all
# simply have no entry in the resulting keyset, which is correct: there's
# nothing to judge "rock-like" on.
_has_len = rock_presel["reco_longest_track_length"].notna().values
rock_presel_candidates = rock_presel.loc[_valid_dir & _has_len]
n_rock_presel_no_candidate = int((~(_valid_dir & _has_len)).sum())
_best_idx = rock_presel_candidates.groupby(
    ["Run", "Subrun", "Evt"])["reco_longest_track_length"].idxmax()
rock_presel_best = rock_presel_candidates.loc[_best_idx]
rock_presel_best_is_rocklike = rock_presel_best["_cos_theta_y"] > ROCK_ANGULAR_VALLEY
rock_rocklike_keyset = set(map(tuple,
    rock_presel_best.loc[rock_presel_best_is_rocklike, ["Run", "Subrun", "Evt"]].values))

n_rock_before = len(rock)
rock_row_keys = list(map(tuple, rock[["Run", "Subrun", "Evt"]].values))
rock_is_fv       = np.array([k in rock_fv_keyset       for k in rock_row_keys])
rock_is_rocklike = np.array([k in rock_rocklike_keyset for k in rock_row_keys])
rock = rock[rock_is_rocklike].reset_index(drop=True)

print(f"\nRestricting rock/{TREE} to FV-contained AND angularly rock-like interactions "
      f"(FV via rock/preselection's fiducial_cut; angle via longest_track "
      f"Cos(theta_LTYD) > {ROCK_ANGULAR_VALLEY}, the numeric valley from the angular-"
      f"separation cell):")
print(f"  rock/{TREE} rows before restriction              : {n_rock_before}")
print(f"  rock/{TREE} rows passing FV alone                 : {rock_is_fv.sum()}  "
      f"({rock_is_fv.sum()/n_rock_before:.1%})")
print(f"  rock/{TREE} rows passing FV + angular cut         : {len(rock)}  "
      f"({len(rock)/n_rock_before:.1%} kept, {1 - len(rock)/n_rock_before:.1%} removed as "
      f"cosmic-like or lacking any qualifying FV interaction)")

# ------------------------------------------------- PPFX weight via join ---
def compute_ppfx(df):
    """PPFX CV weight per row, numu/numubar only; everything else stays 1.0.
    Identical logic to the compute_ppfx used for the `preselection` tree
    elsewhere in this notebook."""
    pdg  = df["true_pdg"].values
    nu_e = df["true_neutrino_energy"].values
    w = np.ones(len(pdg))
    for target_pdg, values, edges in ((14, numu_values, numu_edges),
                                      (-14, numubar_values, numubar_edges)):
        m = (pdg == target_pdg) & np.isfinite(nu_e)
        if m.sum() == 0:
            continue
        e = nu_e[m]
        in_hist = (e >= edges[0]) & (e < edges[-1])
        idx = np.clip(np.searchsorted(edges, e, side='right') - 1, 0, len(values) - 1)
        w[np.where(m)[0]] = np.where(in_hist, values[idx], 1.0)
    return w


print(f"\nBuilding (Run, Subrun, Evt) -> (true_pdg, true_neutrino_energy) lookup "
      f"from events/{MC_SAMPLE}/preselection for the PPFX join...")
presel_truth = f[f"events/{MC_SAMPLE}/preselection"].arrays(
    ["true_pdg", "true_neutrino_energy", "Run", "Subrun", "Evt"], library="pd")
presel_truth = presel_truth[np.isfinite(presel_truth["true_pdg"].values)]
print(f"  {len(presel_truth)} truth-matched rows before dedup")

grp = presel_truth.groupby(["Run", "Subrun", "Evt"])
n_incons = int((grp["true_pdg"].nunique() > 1).sum())
if n_incons > 0:
    print(f"  WARNING: {n_incons} (Run,Subrun,Evt) keys have more than one distinct "
          f"true_pdg value -- taking the first row for those, not silently averaging.")
lookup = presel_truth.drop_duplicates(subset=["Run", "Subrun", "Evt"], keep="first")
print(f"  {len(lookup)} unique (Run,Subrun,Evt) keys in the lookup")

mc_j = mc.merge(lookup[["Run", "Subrun", "Evt", "true_pdg", "true_neutrino_energy"]],
                on=["Run", "Subrun", "Evt"], how="left")
n_matched = int(np.isfinite(mc_j["true_pdg"].values).sum())
print(f"  matched {n_matched} / {len(mc_j)} nominal trigger_time rows "
      f"({n_matched/len(mc_j):.1%}) to a fiducial-volume interaction in preselection.")
print(f"  the remaining {len(mc_j)-n_matched} rows ({(len(mc_j)-n_matched)/len(mc_j):.1%}) "
      f"get PPFX weight 1.0 (no fiducial-volume interaction to look up a true "
      f"neutrino energy/flavor from) -- not reweighted, not excluded.")

mc_j["ppfx_weight"] = compute_ppfx(mc_j)
print(f"  mean PPFX weight (nominal, whole trigger_time tree): {mc_j['ppfx_weight'].mean():.4f}")
mc = mc_j
# Rock is not reweighted (standing instruction), so rock keeps weight 1.0 implicitly.

# --------------------------------------------------------------- helpers ---
# The min value of event_trigger_within_gate is exactly
# -1.7976931348623157e+308 = -DBL_MAX for a large fraction of MC rows and
# ZERO data rows -- a placeholder for "no value" (trigger_emulation_cut only
# requires global_trigger_det_time to be non-NaN, a DIFFERENT field), not a
# real trigger time. Excluded from the histogram and reported separately.
TRIGGER_SENTINEL_THRESHOLD = -1.0e10   # real values sit in roughly [-2, 12]

def _hist_with_overflow(values, weights, bins, valid=None):
    values  = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)

    good = np.isfinite(values)
    if valid is not None:
        good &= np.asarray(valid, dtype=bool)

    v, w = values[good], weights[good]
    n_bad = int((~good).sum())
    sumw_bad, sumw2_bad = weights[~good].sum(), (weights[~good] ** 2).sum()

    n_under = int((v < bins[0]).sum())
    n_over  = int((v > bins[-1]).sum())
    v = np.clip(v, bins[0], np.nextafter(bins[-1], bins[0]))
    sumw,  _ = np.histogram(v, bins=bins, weights=w)
    sumw2, _ = np.histogram(v, bins=bins, weights=w * w)
    return dict(sumw=sumw, sumw2=sumw2, n_under=n_under, n_over=n_over,
                n_good=int(good.sum()), n_bad=n_bad,
                sumw_bad=sumw_bad, sumw2_bad=sumw2_bad)


def _ratio_err(d, p, pe):
    if p <= 0:
        return np.nan
    return (d / p) * np.sqrt(1.0 / max(d, 1.0) + (pe / p) ** 2)


# --------------------------------------------------------------- plotting ---
# Bin width 0.2 us, range -2 to 12 us, matching Howard Fig. 7's own axis
# ("Trigger time (us)", "Events / 0.2 us", range -2 to 12) -- read directly
# off the technote figure, not chosen independently.
BINS = np.linspace(-2.0, 12.0, 71)
bin_centers = 0.5 * (BINS[:-1] + BINS[1:])
bin_width   = BINS[1:] - BINS[:-1]

v_mc, v_rock, v_off, v_on = (mc[TRIGGER_BRANCH].values, rock[TRIGGER_BRANCH].values,
                             offbeam[TRIGGER_BRANCH].values, onbeam[TRIGGER_BRANCH].values)
w_mc_ppfx    = mc["ppfx_weight"].values
w_mc_no_ppfx = np.ones(len(mc))

def _valid(v):
    return np.asarray(v, dtype=float) > TRIGGER_SENTINEL_THRESHOLD

pot_str = f"NuMI {ONBEAM_POT/1e19:.2f}" + r"$\times 10^{19}$ POT"


def run_trigger_time_case(include_rock, case_label, output_suffix, show_label, use_ppfx=True):
    """Build and plot one version of the trigger-time comparison: POT-normalized
    and area-normalized side by side. Nominal MC is PPFX weighted via the join
    above; rock (if included) is not.

    include_rock=False -> Howard Fig. 7 style: nu+cosmic sim + offbeam only.
    include_rock=True  -> rock added as its own component, using the full
                          FV-restricted rock sample with no truth-matching
                          filter -- Roy's approach (see header comment).
    show_label -> whether to draw the case_label as a suptitle / corner note
                  (off for the rock-excluded case, per instruction).
    use_ppfx -> True applies the PPFX flux reweight to the nominal MC (via
                the join above); False instead weights nominal MC by 1.0
                (still POT-scaled by mc_scale), so the two cases isolate the
                effect of the PPFX reweight itself.
    """
    w_mc  = w_mc_ppfx if use_ppfx else w_mc_no_ppfx
    h_mc  = _hist_with_overflow(v_mc,  w_mc * mc_scale, BINS, _valid(v_mc))
    h_off = _hist_with_overflow(v_off, np.ones(len(v_off)), BINS, _valid(v_off))
    h_dat = _hist_with_overflow(v_on,  np.ones(len(v_on)), BINS, _valid(v_on))

    mc_h  = h_mc['sumw']
    off_h = h_off['sumw'] * offbeam_scale
    off_var = h_off['sumw'] * offbeam_scale ** 2
    dat_h = h_dat['sumw']
    dat_err = np.sqrt(dat_h)
    n_data = int(dat_h.sum())

    if include_rock:
        h_rk = _hist_with_overflow(v_rock, np.full(len(v_rock), rock_scale), BINS, _valid(v_rock))
        rk_h = h_rk['sumw']
        rk_sumw2, rk_n_under, rk_n_over, rk_n_bad, rk_sumw_bad = (
            h_rk['sumw2'], h_rk['n_under'], h_rk['n_over'], h_rk['n_bad'], h_rk['sumw_bad'])
    else:
        h_rk = None
        rk_h = np.zeros_like(mc_h)
        rk_sumw2 = np.zeros_like(mc_h)
        rk_n_under = rk_n_over = rk_n_bad = 0
        rk_sumw_bad = 0.0

    pred   = mc_h + rk_h + off_h
    pred_e = np.sqrt(h_mc['sumw2'] + rk_sumw2 + off_var)

    tot_p = pred.sum()
    mc_a, rk_a, off_a = mc_h / tot_p, rk_h / tot_p, off_h / tot_p
    pred_a, pred_a_e  = pred / tot_p, pred_e / tot_p
    dat_a, dat_a_err  = dat_h / n_data, dat_err / n_data

    u_tot = h_mc['n_under'] + rk_n_under + h_off['n_under'] + h_dat['n_under']
    o_tot = h_mc['n_over']  + rk_n_over  + h_off['n_over']  + h_dat['n_over']

    fig = plt.figure(figsize=(12, 5))
    gs  = gridspec.GridSpec(2, 2, height_ratios=[3, 1], hspace=0.05, wspace=0.3)
    tops = [fig.add_subplot(gs[0, i]) for i in range(2)]
    bots = [fig.add_subplot(gs[1, i], sharex=tops[i]) for i in range(2)]

    def draw_panel(ax, mcv, rkv, offv, predv, dv, derr, ylabel, pot_str_,
                  corner=None, show_counts=True):
        # Stacking order (bottom to top): rock (if included), then offbeam,
        # then nu+cosmic sim on top -- so top-to-bottom reads blue, yellow,
        # green, per instruction.
        base = np.zeros(len(BINS) - 1)
        if include_rock:
            ax.bar(bin_centers, rkv,  width=bin_width, bottom=base, color="seagreen",
                   alpha=0.7, label="Rock muons")
            base = base + rkv
        ax.bar(bin_centers, offv, width=bin_width, bottom=base, color="orange",
               alpha=0.7, label="Offbeam (in-time cosmic)")
        base = base + offv
        ax.bar(bin_centers, mcv,  width=bin_width, bottom=base, color="#003087",
               alpha=0.7, label=r"$\nu$ + cosmic sim. (PPFX)")

        # Total stacked height at bin j = predv[j] (mc+rock+offbeam) regardless
        # of stacking order, so use predv directly rather than `base` (which
        # now depends on which component was drawn last).
        for j, n_extra in ((0, u_tot), (len(BINS) - 2, o_tot)):
            if n_extra > 0:
                ax.bar(bin_centers[j], predv[j], width=bin_width[j],
                       fill=False, edgecolor='black', hatch='///', linewidth=0.0, zorder=4)

        ax.step(np.append(BINS[:-1], BINS[-1]), np.append(predv, predv[-1]),
                where='post', color='black', linewidth=1.5,
                label=(f"Prediction ({predv.sum():.1f})" if show_counts else "Prediction (Area Norm.)"))
        ax.errorbar(bin_centers, dv, yerr=derr, fmt='o', markersize=4,
                    markerfacecolor="black", markeredgecolor="black", color="black",
                    capsize=3, capthick=1.5, elinewidth=1.5,
                    label=(f"Data ({n_data})" if show_counts else "Data (Area Norm.)"), zorder=5)

        ax.set_ylabel(ylabel, fontsize=11, fontfamily="sans-serif", fontweight='bold')
        ax.grid(True, alpha=0.3)
        apply_tick_style(ax)
        plt.setp(ax.get_xticklabels(), visible=False)
        ax.set_ylim(0, ax.get_ylim()[1] * 1.5)
        # Legend order matches draw order (rock first if present, then
        # offbeam, then nu+cosmic sim, then prediction/data).
        h, l = ax.get_legend_handles_labels()
        ax.legend(h, l, fontsize=8, loc='upper left', framealpha=0.0, edgecolor='none')
        add_icarus_label(ax)
        add_pot_label(ax, pot_str_)
        if corner:
            add_corner_label(ax, corner)

    def draw_ratio(ax, predv, dv, derr):
        with np.errstate(invalid="ignore", divide="ignore"):
            r    = np.where(predv > 0, dv / predv, np.nan)
            rerr = np.where(predv > 0, derr / predv, np.nan)
        inside = np.isfinite(r) & (r >= 0.0) & (r <= 2.0)
        ax.errorbar(bin_centers[inside], r[inside], yerr=rerr[inside],
                    fmt='o', markersize=4, markerfacecolor='black', markeredgecolor='black',
                    color='black', capsize=3, capthick=1.5, elinewidth=1.5)
        hi = np.isfinite(r) & (r > 2.0)
        if hi.any():
            ax.plot(bin_centers[hi], np.full(hi.sum(), 1.92), marker='^', linestyle='none',
                    markersize=5, color='red')
        ax.axhline(1.0, color='black', linestyle='--', linewidth=1.0, alpha=0.8)
        ax.set_ylim(0.0, 2.0)
        ax.set_ylabel(r"$\mathbf{Data/MC}$", fontsize=11)
        ax.set_xlabel(r"$\mathbf{Trigger\ Time\ [\mu s]}$", fontsize=11)
        ax.grid(True, alpha=0.3)
        apply_tick_style(ax)

    corner_text = (r'$\mathbf{PPFX\ REWEIGHT\ APPLIED}$' if use_ppfx
                   else r'$\mathbf{NO\ PPFX\ REWEIGHT}$') + (f"\n{case_label}" if show_label else "")

    draw_panel(tops[0], mc_h, rk_h, off_h, pred, dat_h, dat_err,
              r"$\mathbf{Events\ /\ 0.2\ \mu s}$", pot_str, corner=corner_text)
    draw_ratio(bots[0], pred, dat_h, dat_err)

    draw_panel(tops[1], mc_a, rk_a, off_a, pred_a, dat_a, dat_a_err,
              r"$\mathbf{Fraction\ /\ 0.2\ \mu s}$", "Area Normalized",
              corner=corner_text, show_counts=False)
    draw_ratio(bots[1], pred_a, dat_a, dat_a_err)

    if show_label:
        fig.suptitle(case_label, fontsize=11, fontweight='bold', y=1.03)

    os.makedirs(os.path.join(OUTPUT_DIR, "pdf"),  exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, "jpeg"), exist_ok=True)
    fig.savefig(os.path.join(OUTPUT_DIR, f"pdf/preselection_trigger_time_{output_suffix}.pdf"),
                bbox_inches="tight", dpi=300)
    fig.savefig(os.path.join(OUTPUT_DIR, f"jpeg/preselection_trigger_time_{output_suffix}.jpeg"),
                bbox_inches="tight", dpi=300)
    plt.show()
    plt.close(fig)

    ppfx_str = "PPFX applied" if use_ppfx else "no PPFX"
    print(f"\n--- trigger_time [{case_label}] (POT-normalized | area-normalized, {ppfx_str}) ---")
    n_rock_used = len(v_rock) if include_rock else 0
    print(f"  rows loaded: mc {len(mc)}  rock {n_rock_used} (0 if excluded)  "
          f"onbeam {len(onbeam)}  offbeam {len(offbeam)}")
    print(f"  data               : {n_data}")
    print(f"  prediction         : {pred.sum():.1f} +/- {np.sqrt((pred_e**2).sum()):.1f} (stat)")
    print(f"    nu+cosmic sim    : {mc_h.sum():.1f}")
    if include_rock:
        print(f"    rock muons       : {rk_h.sum():.1f}  (full FV-restricted rock sample, POT-scaled, "
              f"no truth-matching filter -- Roy's approach, see header comment)")
    else:
        print(f"    rock muons       : excluded from this case")
    print(f"    offbeam (cosmic) : {off_h.sum():.1f}")
    print(f"  data/MC            : {n_data/pred.sum():.4f} +/- "
          f"{_ratio_err(n_data, pred.sum(), np.sqrt((pred_e**2).sum())):.4f}")
    print(f"  folded into edge bins: underflow {u_tot}, overflow {o_tot}")

    print(f"\n  excluded as 'no usable {TRIGGER_BRANCH}' (<= {TRIGGER_SENTINEL_THRESHOLD:.0e}, "
          f"i.e. the -DBL_MAX sentinel):")
    # `scale` here is whatever still needs to be applied to h['sumw_bad'] to
    # get a properly POT/livetime-scaled excluded weight. For MC and rock,
    # h['sumw_bad'] is summed from a weight array that ALREADY has mc_scale /
    # rock_scale baked in (see how h_mc / h_rk are built above), so the
    # remaining factor is 1.0. Offbeam is the opposite case: h_off is built
    # from raw, unscaled weights (weight=1), so offbeam_scale still needs to
    # be applied here for this line to mean what its label says.
    rows = [(MC_SAMPLE, h_mc, len(mc), 1.0),
            ("onbeam", h_dat, len(onbeam), 1.0),
            ("offbeam", h_off, len(offbeam), offbeam_scale)]
    if include_rock:
        rows.insert(1, ("rock", h_rk, len(v_rock), 1.0))
    for name, h, n_rows, scale in rows:
        frac = h['n_bad'] / n_rows if n_rows else float('nan')
        print(f"    {name:>8}: {h['n_bad']:7d} / {n_rows:7d} rows ({frac:.1%})  "
              f"-- scaled weight excluded: {h['sumw_bad'] * scale:.1f}")

    return dict(n_data=n_data, pred=pred.sum(), pred_e=float(np.sqrt((pred_e**2).sum())),
                # Bin-level arrays, needed for the residual-shape and
                # out-of-window diagnostics below -- not used by the plot
                # itself, which only needs the scalar summary above.
                dat_h=dat_h, pred_h=pred, mc_h=mc_h, off_h=off_h, rk_h=rk_h)


# Rock included, using Roy's approach: the full FV-restricted rock sample
# (angularly cut, see the restriction block above), POT-scaled, no
# truth-matching filter on top. This is now the only case plotted --
# rock-excluded and no-PPFX variants were dropped per instruction (they
# were comparison/diagnostic aids, not part of the final plot).
result_with_rock = run_trigger_time_case(
    include_rock=True, case_label="Rock included (Roy's approach: full FV-restricted rock sample, POT-scaled)",
    output_suffix="with_rock", show_label=False, use_ppfx=True)

# ============================================================================
# Diagnostic 1 (out-of-window cosmic-normalization check): outside the NuMI
# beam window (t < 0 or t > 9.6 us, the flash_low_t/flash_high_t window
# already used elsewhere in this toml) there is no beam-induced rock-muon
# component -- only cosmics. Comparing data to (nu+cosmic sim + offbeam),
# ROCK EXCLUDED, restricted to that region tests the cosmic normalization
# (mc_scale, offbeam_scale) independently of anything rock-related. If this
# doesn't check out, no rock fix will make the in-window comparison correct
# either -- the cosmic normalization has to be right first.
# ============================================================================
# Diagnostic 1/1b need the rock-excluded, PPFX-applied binned prediction,
# but that case is no longer plotted (dropped per instruction). Recompute
# just the arrays here -- same formula as run_trigger_time_case's
# include_rock=False path -- without calling the plotting function.
_h_mc_nr  = _hist_with_overflow(v_mc,  w_mc_ppfx * mc_scale, BINS, _valid(v_mc))
_h_off_nr = _hist_with_overflow(v_off, np.ones(len(v_off)), BINS, _valid(v_off))
_h_dat_nr = _hist_with_overflow(v_on,  np.ones(len(v_on)), BINS, _valid(v_on))
_mc_h_nr  = _h_mc_nr['sumw']
_off_h_nr = _h_off_nr['sumw'] * offbeam_scale
_dat_h_nr = _h_dat_nr['sumw']
r0 = dict(dat_h=_dat_h_nr, pred_h=_mc_h_nr + _off_h_nr, mc_h=_mc_h_nr, off_h=_off_h_nr)

out_of_window = (bin_centers < 0.0) | (bin_centers > 9.6)
data_oow = r0['dat_h'][out_of_window].sum()
pred_oow = r0['pred_h'][out_of_window].sum()
mc_oow   = r0['mc_h'][out_of_window].sum()
off_oow  = r0['off_h'][out_of_window].sum()
print(f"\n=========== Diagnostic 1: out-of-window cosmic normalization "
      f"(t < 0 or t > 9.6 us, rock excluded) ===========")
print(f"  data                    : {data_oow:.1f}")
print(f"  prediction (mc + offbeam): {pred_oow:.1f}  (nu+cosmic sim {mc_oow:.1f} + offbeam {off_oow:.1f})")
if pred_oow > 0:
    print(f"  data/prediction         : {data_oow/pred_oow:.4f}  "
          f"(1.0 would mean the cosmic normalization is sound out here; "
          f"a large deviation means mc_scale/offbeam_scale need attention "
          f"BEFORE any rock fix, since rock contributes ~nothing to this region)")

# ----------------------------------------------------------------------
# Diagnostic 1b: is the out-of-window deficit FLAT (-> a missing overall
# scale factor on mc_scale/offbeam_scale, same cause everywhere) or
# EDGE-CONCENTRATED (-> something specific to the trigger-emulation cut's
# behavior near the gate boundary, different cause)? Split pre-gate
# (t < 0) from post-gate (t > 9.6) separately, since a pre-spill vs
# post-spill asymmetry would itself be informative, and plot the
# bin-by-bin data/prediction ratio across the full range so the window
# edges are visible rather than only summarized.
# ----------------------------------------------------------------------
pre_gate  = bin_centers < 0.0
post_gate = bin_centers > 9.6
for label, mask in (("pre-gate (t < 0 us)", pre_gate), ("post-gate (t > 9.6 us)", post_gate)):
    d = r0['dat_h'][mask].sum()
    p = r0['pred_h'][mask].sum()
    ratio = d / p if p > 0 else float('nan')
    print(f"    {label:24s}: data {d:.1f}  pred {p:.1f}  data/pred {ratio:.4f}")

# Diagnostic 1b plot removed per instruction -- pre-gate/post-gate numeric
# breakdown above is kept.

# ============================================================================
# Diagnostic 2 (residual-shape test, decides normalization-only vs
# contaminated-population for rock): compare the SHAPE of
# data - (nu+cosmic sim + offbeam), rock excluded, against the SHAPE of the
# rock sample itself, both area-normalized. If the shapes agree, the rock
# POT/scale is what's wrong (normalization only) -- rescale and move on. If
# they disagree (e.g. rock has the t=0 spike from double-counted
# pre-spill-cosmic triggers that the residual doesn't have), the rock
# POPULATION is contaminated and a truth-matching-style filter is needed on
# top of any rescaling.
# ============================================================================
residual = r0['dat_h'] - r0['pred_h']   # data - (mc+offbeam), rock excluded
residual_pos = np.clip(residual, 0.0, None)   # only the "needs something added" part is meaningful as a shape
residual_shape = residual_pos / residual_pos.sum() if residual_pos.sum() > 0 else residual_pos

h_rock_raw = _hist_with_overflow(v_rock, np.ones(len(v_rock)), BINS, _valid(v_rock))
rock_counts = h_rock_raw['sumw']
rock_shape = rock_counts / rock_counts.sum() if rock_counts.sum() > 0 else rock_counts

# Simple shape-agreement metric: cosine similarity between the two
# normalized shape vectors (1.0 = identical shape, 0.0 = orthogonal).
cos_sim = float(np.dot(residual_shape, rock_shape) /
                (np.linalg.norm(residual_shape) * np.linalg.norm(rock_shape) + 1e-300))
print(f"\n=========== Diagnostic 2: residual shape (data - mc - offbeam) vs "
      f"rock shape (both area-normalized) ===========")
print(f"  residual (positive part) total : {residual_pos.sum():.1f} events "
      f"(residual before clipping negative bins to 0: {residual.sum():.1f})")
print(f"  cosine similarity of shapes     : {cos_sim:.4f}  "
      f"(close to 1.0 -> shapes agree, points to a pure normalization/POT "
      f"problem with rock; well below 1.0 -> the rock population's shape "
      f"doesn't match what's missing from data, points to a contaminated "
      f"rock population needing a filter, not just a rescale)")

# Diagnostic 2 plots removed per instruction -- cosine-similarity numbers
# above are kept, and residual_pos/rock_counts/rock_shape/cos_sim (computed
# above) are still needed by Diagnostic 2b below.

# ============================================================================
# Diagnostic 2b: quantify the pre-spill-cosmic spike bin Howard's technote
# describes ("an additional component here of simulated cosmic muons from
# before the beam spill providing the trigger ... the spike right near t=0
# in this sample, which may have some double-counting effect with the
# in-time data sample"). Identified here as the bin where rock's raw shape
# most exceeds its own local neighborhood -- not manually picked -- then:
# (a) what fraction of rock's total POT-scaled contribution sits in that one
# bin, and (b) does excluding it from the shape comparison raise the cosine
# similarity, confirming it (not the whole rock population) is the outlier.
# ============================================================================
spike_bin = int(np.argmax(rock_shape))
neighbor_avg = (rock_shape[max(spike_bin - 3, 0):spike_bin].sum() +
                rock_shape[spike_bin + 1:spike_bin + 4].sum()) / 6.0
spike_frac_of_rock = rock_counts[spike_bin] / rock_counts.sum()
rock_scaled_spike = rock_counts[spike_bin] * rock_scale  # same units as rk_h

mask_no_spike = np.ones(len(BINS) - 1, dtype=bool)
mask_no_spike[spike_bin] = False
res_ns = residual_pos[mask_no_spike]
rock_ns = rock_counts[mask_no_spike]
res_ns_shape = res_ns / res_ns.sum() if res_ns.sum() > 0 else res_ns
rock_ns_shape = rock_ns / rock_ns.sum() if rock_ns.sum() > 0 else rock_ns
cos_sim_ns = float(np.dot(res_ns_shape, rock_ns_shape) /
                   (np.linalg.norm(res_ns_shape) * np.linalg.norm(rock_ns_shape) + 1e-300))

print(f"\n=========== Diagnostic 2b: isolating the pre-spill-cosmic spike bin ===========")
print(f"  spike bin center                        : {bin_centers[spike_bin]:.2f} us  "
      f"(rock shape value {rock_shape[spike_bin]:.4f} vs local neighbor average {neighbor_avg:.4f}, "
      f"{rock_shape[spike_bin]/max(neighbor_avg,1e-12):.1f}x)")
print(f"  fraction of rock's raw (unscaled) counts in this bin : {spike_frac_of_rock:.1%}")
rock_total_scaled = rock_counts.sum() * rock_scale
print(f"  POT-scaled rock contribution from this bin alone     : {rock_scaled_spike:.1f} events  "
      f"(out of {rock_total_scaled:.1f} total rock contribution, {rock_scaled_spike/rock_total_scaled:.1%})")
print(f"  cosine similarity, ALL bins    : {cos_sim:.4f}  (from Diagnostic 2)")
print(f"  cosine similarity, spike EXCLUDED : {cos_sim_ns:.4f}  "
      f"(higher -> confirms the spike bin specifically is the outlier, not the whole "
      f"rock shape; if this is still well below 1.0, the mismatch is broader than just "
      f"the one bin and a single-bin trim will not be enough)")

print("\nDone.")


CELL VERSION: trigger_time_v22_trimmed_plots_2026-08-20
Reading exposures (trigger_time tree)...
    onbeam: POT read from events/onbeam/POT histogram = 2.216270e+19
    onbeam: Livetime read from events/onbeam/Livetime histogram = 4.839580e+05
    offbeam: POT read from events/offbeam/POT histogram = 0.000000e+00
    offbeam: Livetime read from events/offbeam/Livetime histogram = 5.078498e+06
    rock: POT read from events/rock/POT histogram = 1.706694e+19
    rock: Livetime read from events/rock/Livetime histogram = 2.017200e+05
    nominal: POT read from events/nominal/POT histogram = 7.785208e+20
    nominal: Livetime read from events/nominal/Livetime histogram = 1.501200e+06

Onbeam POT (raw, from file)                : 2.216270e+19
Onbeam POT (validated, histogram-integration): 2.361240e+19  (Moreno, NuMI POT Counting Closure Test, Apr 1 2025 -- used for ONBEAM_POT/mc_scale/rock_scale only; NOT applied to offbeam livetime, see comment above)
Offbeam livetime (raw, from file, used

KeyInFileError: not found: 'reco_longest_track_length'
in file /Users/rvizarreta/Library/CloudStorage/GoogleDrive-rvizarreta14@gmail.com/My Drive/🏛 PhD Repository/🚀 Research/🤖 Experiments&Projects/ICARUS/ICARUS_CC0pi_Selection/data/preselection.root
in object /events/rock/preselection;1

In [ ]:
# ============================================================================
# Other preselection-level plots for the technote: flash time and vertex
# x/y/z -- from the `preselection` tree (one row per reconstructed
# interaction), not `trigger_time`. Same file (data/preselection.root),
# same cosmetic conventions as the trigger-time cell: POT-normalized and
# area-normalized side by side, PPFX reweight applied to nominal MC and
# labeled "PPFX REWEIGHT APPLIED" in bold caps, no prediction-uncertainty
# band, no chi2/ndf, no case-label title -- and each variable produced
# twice, rock excluded and rock included.
#
# Unlike `trigger_time`, `preselection` is a mode = "reco" tree, so it
# carries true_pdg / true_neutrino_energy directly -- no Run/Subrun/Evt join
# needed here, PPFX is computed the same direct way as in the trigger-time
# cell.
#
# Per-variable masks (require_flash_match, valid_fn): flash_time is plotted
# with no mask on itself; the vertex coordinates require an in-time flash
# match (0 < reco_flash_time < 9.6 us) since they are not meaningful
# without one.
#
# Rock muon treatment: matches the trigger-time cell (v21) -- the rock
# sample is POT-scaled to data exposure, but restricted first to rows whose
# longest reconstructed track is angularly rock-like (Cos(theta_LTYD) =
# longest_track start_dir_y > 0.060, the numeric valley found in the
# "rock-vs-cosmic angular separation" cell). That cell established this is
# necessary: the raw rock production carries a large standalone-cosmic
# contamination (per Howard's Fig. 8 note), and neither of the two
# truth-level cross-checks tried (n_true_nu, true_category) could isolate
# it -- SPINE fails to truth-match rock-origin vertices to a neutrino
# almost universally, for genuine rock muons and standalone cosmics alike,
# since a rock interaction's true vertex is outside the TPC either way.
# The FV-volume requirement (fiducial_cut, already built into this
# `preselection` tree by construction) turned out to be a near no-op on
# its own for the same geometric reason. See the angular-separation cell
# for the full derivation of the 0.060 valley.
#
# v3: preselection.root was rebuilt with crtpmt_veto added to the
# `preselection` tree's cut list (toml change, applies to onbeam/offbeam/MC
# and rock alike, since they all read the same tree). No logic change is
# needed here -- the cut is baked in at production level -- but the plots
# are now annotated "CRT-PMT SPILL CUT APPLIED" alongside the PPFX label.
# Note the rock sample's Cos(theta_LTYD) valley was derived pre-crtpmt_veto;
# ROCK_ANGULAR_VALLEY = 0.060 is kept as-is for now, per instruction.
#
# Self-contained: re-imports and re-defines everything it needs.
# ============================================================================

import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import uproot

print("CELL VERSION: preselection_plots_v3_crtpmt_veto_2026-08-20")

# ------------------------------------------------------------------ config --
DATA_FILE = ("/Users/rvizarreta/Library/CloudStorage/GoogleDrive-rvizarreta14@gmail.com/"
             "My Drive/\U0001f3db PhD Repository/\U0001f680 Research/"
             "\U0001f916 Experiments&Projects/ICARUS/ICARUS_CC0pi_Selection/"
             "data/preselection.root")

FLUX_FILE = ("/Users/rvizarreta/Library/CloudStorage/GoogleDrive-rvizarreta14@gmail.com/"
             "My Drive/\U0001f3db PhD Repository/\U0001f680 Research/"
             "\U0001f916 Experiments&Projects/ICARUS/ICARUS_CC0pi_Selection/"
             "systematics/2025-04-08_out_450.37_7991.98_79512.66.root")

OUTPUT_DIR = ("/Users/rvizarreta/Library/CloudStorage/GoogleDrive-rvizarreta14@gmail.com/"
              "My Drive/\U0001f3db PhD Repository/\U0001f680 Research/"
              "\U0001f916 Experiments&Projects/ICARUS/ICARUS_CC0pi_Selection/"
              "ICARUS-NuMI-CC0pi-Selection/spineplot/myAnalysis/"
              "1muNp0pi_Nge1_uncontained/preselection")

ONBEAM_POT  = 2.36124e19
MC_SAMPLE   = "nominal"
TREE        = "preselection"
horn_current = 'fhc'
# nominal_pot is read from the file itself below (same reasoning as the
# trigger_time cell): this preselection.root was built from a partial job
# set, so its generated POT is not the 7.178e20 used for the older file.

# ------------------------------------------------------------------ style ---
def apply_tick_style(ax):
    ax.tick_params(axis='both', which='major', labelsize=10, size=8, width=2, direction='in')
    ax.minorticks_on()
    ax.tick_params(axis='both', which='minor', size=4, width=1, direction='in')
    for lbl in ax.get_xticklabels() + ax.get_yticklabels():
        lbl.set_fontfamily('sans-serif')
    ax.yaxis.get_offset_text().set_fontfamily('sans-serif')
    ax.xaxis.get_offset_text().set_fontfamily('sans-serif')

def add_icarus_label(ax, subtitle="Work-in-Progress"):
    yr, xr = ax.get_ylim(), ax.get_xlim()
    ax.text(xr[0] + 0.01 * (xr[1] - xr[0]), yr[1] + 0.02 * (yr[1] - yr[0]),
            r'$\bf{ICARUS \cdot NuMI}$ Data (10% Run 2)' + '\n' + subtitle,
            fontsize=8, color="chocolate", verticalalignment="bottom",
            fontfamily="sans-serif")

def add_pot_label(ax, pot_str):
    yr, xr = ax.get_ylim(), ax.get_xlim()
    ax.text(xr[1] - 0.01 * (xr[1] - xr[0]), yr[1] + 0.02 * (yr[1] - yr[0]), pot_str,
            fontsize=8, color="black", verticalalignment="bottom",
            horizontalalignment="right", fontfamily="sans-serif")

def add_corner_label(ax, text, y=0.85):
    ax.text(0.97, y, text, transform=ax.transAxes, fontsize=7, color='black',
            horizontalalignment='right', verticalalignment='top',
            fontfamily="sans-serif")

# ------------------------------------------------------------------- load ---
f = uproot.open(DATA_FILE)
flux = uproot.open(FLUX_FILE)
ppfx_numu    = flux[f'ppfx_flux_weights/hweights_{horn_current}_numu;1']
ppfx_numubar = flux[f'ppfx_flux_weights/hweights_{horn_current}_numubar;1']
numu_values, numu_edges       = ppfx_numu.values(),    ppfx_numu.axes[0].edges()
numubar_values, numubar_edges = ppfx_numubar.values(), ppfx_numubar.axes[0].edges()


def read_exposure(sample, tree=TREE):
    pot = lt = None
    if f"events/{sample}/POT" in f:
        pot = f[f"events/{sample}/POT"].values().sum()
    if f"events/{sample}/Livetime" in f:
        lt = f[f"events/{sample}/Livetime"].values().sum()
    if pot is not None or lt is not None:
        return pot, lt
    key = f"events/{sample}/{tree}_exposure"
    if key not in f:
        raise KeyError(f"no exposure information found for sample '{sample}' "
                       f"(looked for events/{sample}/POT, /Livetime and {key})")
    t = f[key]
    print(f"    {sample}/{tree}_exposure branches: {t.keys()}")
    arrays = t.arrays(library="np")
    for name, vals in arrays.items():
        low = name.lower()
        if pot is None and "pot" in low:
            pot = float(np.nansum(vals))
        if lt is None and ("livetime" in low or "live_time" in low):
            lt = float(np.nansum(vals))
    return pot, lt


print("Reading exposures (preselection tree)...")
_,           onbeam_lt  = read_exposure("onbeam")
_,           offbeam_lt = read_exposure("offbeam")
rock_pot,        _      = read_exposure("rock")
nominal_pot_raw, _      = read_exposure(MC_SAMPLE)

for label, val in (("onbeam livetime", onbeam_lt), ("offbeam livetime", offbeam_lt),
                   ("rock POT", rock_pot), ("nominal MC POT (raw)", nominal_pot_raw)):
    if val is None or not np.isfinite(val) or val <= 0:
        raise ValueError(f"{label} came back as {val}. Check the exposure branch "
                         f"names printed above.")

# Correction for broken/recovered job files: nominal_pot_raw, read from the
# file's own events/nominal/POT histogram, matches (to 5 sig figs) the raw
# COMPLETE-production POT value (7.78520e+20) that this correction was
# derived from -- raw 7.78520e+20 -> corrected (true, recovered-file-loss-
# accounted) POT 7.178390e+20 (confirmed against the same file in the
# trigger-time cell).
POT_CORRECTION_FACTOR = 7.178390e20 / 7.78520e20
nominal_pot = nominal_pot_raw * POT_CORRECTION_FACTOR

mc_scale      = ONBEAM_POT / nominal_pot
offbeam_scale = onbeam_lt / offbeam_lt
rock_scale    = ONBEAM_POT / rock_pot

print(f"Nominal MC POT (raw, from file) : {nominal_pot_raw:.4e}")
print(f"POT correction factor           : {POT_CORRECTION_FACTOR:.6f}  "
      f"(= 7.178390e+20 / 7.78520e+20)")
print(f"Nominal MC POT (corrected)      : {nominal_pot:.4e}")
print(f"MC scale:      {mc_scale:.4e}")
print(f"Offbeam scale: {offbeam_scale:.4e}")
print(f"Rock scale:    {rock_scale:.4e}")

print(f"\nLoading '{TREE}' trees...")
mc      = f[f"events/{MC_SAMPLE}/{TREE}"].arrays(library="pd")
rock    = f[f"events/rock/{TREE}"].arrays(library="pd")
onbeam  = f[f"events/onbeam/{TREE}"].arrays(library="pd")
offbeam = f[f"events/offbeam/{TREE}"].arrays(library="pd")

for name, df in ((MC_SAMPLE, mc), ("rock", rock), ("onbeam", onbeam), ("offbeam", offbeam)):
    print(f"  {name:>8}: {len(df)} rows")

for col in ("true_pdg", "true_neutrino_energy"):
    if col not in mc.columns:
        raise KeyError(f"'{col}' not found in events/{MC_SAMPLE}/{TREE}; "
                       f"columns = {list(mc.columns)}")

# ------------------------------------ rock: angular (rock-vs-cosmic) cut ---
# Same fix as the trigger-time cell (v21): the full, unfiltered rock/
# preselection sample includes a large standalone-cosmic contamination
# riding along in the rock production (confirmed via the "rock-vs-cosmic
# angular separation" cell -- a clean bimodal Cos(theta_LTYD) distribution,
# numeric valley at 0.060). Unlike the trigger-time cell, no event-level
# join/idxmax logic is needed here: `preselection` is already one row per
# reconstructed interaction, so `reco_longest_track_start_dir_{x,y,z}` on
# THIS row is already the direction of THIS row's own longest track --
# the same row whose vertex_x/y/z/flash_time is what gets plotted below.
# A direct per-row mask is therefore exact, not an approximation the way
# the trigger-time cell's event-level "pick one interaction" logic was.
ROCK_ANGULAR_VALLEY = 0.060
_vx = rock["reco_longest_track_start_dir_x"].values
_vy = rock["reco_longest_track_start_dir_y"].values
_vz = rock["reco_longest_track_start_dir_z"].values
_norm = np.sqrt(_vx**2 + _vy**2 + _vz**2)
_valid_dir = np.isfinite(_norm) & (_norm > 0)
_cos_theta_y = np.full(len(rock), np.nan)
_cos_theta_y[_valid_dir] = _vy[_valid_dir] / _norm[_valid_dir]
_is_rocklike = _valid_dir & (_cos_theta_y > ROCK_ANGULAR_VALLEY)

n_rock_before = len(rock)
rock = rock[_is_rocklike].reset_index(drop=True)
print(f"\nRestricting rock/{TREE} to angularly rock-like interactions "
      f"(longest_track Cos(theta_LTYD) > {ROCK_ANGULAR_VALLEY}, the numeric "
      f"valley from the angular-separation cell):")
print(f"  rock/{TREE} rows before angular cut : {n_rock_before}")
print(f"  rock/{TREE} rows after angular cut  : {len(rock)}  "
      f"({len(rock)/n_rock_before:.1%} kept, {1 - len(rock)/n_rock_before:.1%} removed "
      f"as cosmic-like or lacking a valid longest-track direction)")

# --------------------------------------------------------- PPFX weighting ---
def compute_ppfx(df):
    """PPFX CV weight per row, numu/numubar only; everything else stays 1.0.
    Identical logic to the compute_ppfx used elsewhere in this notebook."""
    pdg  = df["true_pdg"].values
    nu_e = df["true_neutrino_energy"].values
    w = np.ones(len(pdg))
    for target_pdg, values, edges in ((14, numu_values, numu_edges),
                                      (-14, numubar_values, numubar_edges)):
        m = (pdg == target_pdg) & np.isfinite(nu_e)
        if m.sum() == 0:
            continue
        e = nu_e[m]
        in_hist = (e >= edges[0]) & (e < edges[-1])
        idx = np.clip(np.searchsorted(edges, e, side='right') - 1, 0, len(values) - 1)
        w[np.where(m)[0]] = np.where(in_hist, values[idx], 1.0)
    return w

mc["ppfx_weight"] = compute_ppfx(mc)
print(f"Mean PPFX weight ({MC_SAMPLE}, whole tree): {mc['ppfx_weight'].mean():.4f}")
# Rock is NOT PPFX-reweighted (standing instruction: rock muons are not beam
# neutrinos).

# --------------------------------------------------------------- helpers ---
def _hist_with_overflow(values, weights, bins, valid=None):
    values  = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    good = np.isfinite(values)
    if valid is not None:
        good &= np.asarray(valid, dtype=bool)
    v, w = values[good], weights[good]
    n_bad = int((~good).sum())
    sumw_bad, sumw2_bad = weights[~good].sum(), (weights[~good] ** 2).sum()
    n_under = int((v < bins[0]).sum())
    n_over  = int((v > bins[-1]).sum())
    v = np.clip(v, bins[0], np.nextafter(bins[-1], bins[0]))
    sumw,  _ = np.histogram(v, bins=bins, weights=w)
    sumw2, _ = np.histogram(v, bins=bins, weights=w * w)
    return dict(sumw=sumw, sumw2=sumw2, n_under=n_under, n_over=n_over,
                n_good=int(good.sum()), n_bad=n_bad,
                sumw_bad=sumw_bad, sumw2_bad=sumw2_bad)


def _ratio_err(d, p, pe):
    if p <= 0:
        return np.nan
    return (d / p) * np.sqrt(1.0 / max(d, 1.0) + (pe / p) ** 2)


def flash_matched(df):
    return (df["reco_flash_time"] > 0.0) & (df["reco_flash_time"] < 9.6)


# --------------------------------------------------------------- plotting ---
def run_preselection_case(variable, xlabel, bins, output_suffix, include_rock,
                          require_flash_match=False, valid_fn=None):
    """POT-normalized / area-normalized side by side, PPFX applied to nominal,
    rock included or excluded. Same cosmetic conventions as the finalized
    trigger-time cell: no prediction stat-unc band, no chi2/ndf, no title."""
    bins = np.asarray(bins, dtype=float)
    bin_centers = 0.5 * (bins[:-1] + bins[1:])
    bin_width   = bins[1:] - bins[:-1]

    def build_mask(df):
        m = np.ones(len(df), dtype=bool)
        if require_flash_match:
            m &= flash_matched(df).values
        return m

    m_mc, m_rock = build_mask(mc), build_mask(rock)
    m_off, m_on  = build_mask(offbeam), build_mask(onbeam)

    v_mc   = mc.loc[m_mc,       variable].values
    v_rock = rock.loc[m_rock,   variable].values
    v_off  = offbeam.loc[m_off, variable].values
    v_on   = onbeam.loc[m_on,   variable].values

    vf = (lambda v: valid_fn(v)) if valid_fn is not None else (lambda v: None)

    h_mc  = _hist_with_overflow(v_mc,  mc.loc[m_mc, "ppfx_weight"].values * mc_scale, bins, vf(v_mc))
    h_off = _hist_with_overflow(v_off, np.ones(len(v_off)), bins, vf(v_off))
    h_dat = _hist_with_overflow(v_on,  np.ones(len(v_on)),  bins, vf(v_on))

    mc_h  = h_mc['sumw']
    off_h = h_off['sumw'] * offbeam_scale
    off_var = h_off['sumw'] * offbeam_scale ** 2
    dat_h = h_dat['sumw']
    dat_err = np.sqrt(dat_h)
    n_data = int(dat_h.sum())

    if include_rock:
        h_rk = _hist_with_overflow(v_rock, np.full(len(v_rock), rock_scale), bins, vf(v_rock))
        rk_h = h_rk['sumw']
        rk_sumw2 = h_rk['sumw2']
    else:
        h_rk = None
        rk_h = np.zeros_like(mc_h)
        rk_sumw2 = np.zeros_like(mc_h)

    pred   = mc_h + rk_h + off_h
    pred_e = np.sqrt(h_mc['sumw2'] + rk_sumw2 + off_var)

    tot_p = pred.sum()
    mc_a, rk_a, off_a = mc_h / tot_p, rk_h / tot_p, off_h / tot_p
    pred_a = pred / tot_p
    dat_a, dat_a_err  = dat_h / n_data, dat_err / n_data

    u_tot = h_mc['n_under'] + (h_rk['n_under'] if include_rock else 0) + h_off['n_under'] + h_dat['n_under']
    o_tot = h_mc['n_over']  + (h_rk['n_over']  if include_rock else 0) + h_off['n_over']  + h_dat['n_over']

    fig = plt.figure(figsize=(12, 5))
    gs  = gridspec.GridSpec(2, 2, height_ratios=[3, 1], hspace=0.05, wspace=0.3)
    tops = [fig.add_subplot(gs[0, i]) for i in range(2)]
    bots = [fig.add_subplot(gs[1, i], sharex=tops[i]) for i in range(2)]

    def draw_panel(ax, mcv, rkv, offv, predv, dv, derr, ylabel, pot_str_, corner=None, show_counts=True):
        base = np.zeros(len(bins) - 1)
        ax.bar(bin_centers, mcv,  width=bin_width, bottom=base, color="#003087",
               alpha=0.7, label=r"$\nu$ + cosmic sim. (PPFX)")
        base = base + mcv
        if include_rock:
            ax.bar(bin_centers, rkv, width=bin_width, bottom=base, color="seagreen",
                   alpha=0.7, label="Rock muons")
            base = base + rkv
        ax.bar(bin_centers, offv, width=bin_width, bottom=base, color="orange",
               alpha=0.7, label="Offbeam (in-time cosmic)")

        for j, n_extra in ((0, u_tot), (len(bins) - 2, o_tot)):
            if n_extra > 0:
                ax.bar(bin_centers[j], base[j] + offv[j], width=bin_width[j],
                       fill=False, edgecolor='black', hatch='///', linewidth=0.0, zorder=4)

        ax.step(np.append(bins[:-1], bins[-1]), np.append(predv, predv[-1]),
                where='post', color='black', linewidth=1.5,
                label=(f"Prediction ({predv.sum():.1f})" if show_counts else "Prediction (Area Norm.)"))
        ax.errorbar(bin_centers, dv, yerr=derr, fmt='o', markersize=4,
                    markerfacecolor="black", markeredgecolor="black", color="black",
                    capsize=3, capthick=1.5, elinewidth=1.5,
                    label=(f"Data ({n_data})" if show_counts else "Data (Area Norm.)"), zorder=5)

        ax.set_ylabel(ylabel, fontsize=11, fontfamily="sans-serif", fontweight='bold')
        ax.grid(True, alpha=0.3)
        apply_tick_style(ax)
        plt.setp(ax.get_xticklabels(), visible=False)
        ax.set_ylim(0, ax.get_ylim()[1] * 1.5)
        h, l = ax.get_legend_handles_labels()
        ax.legend(h[::-1], l[::-1], fontsize=8, loc='upper left', framealpha=0.0, edgecolor='none')
        add_icarus_label(ax)
        add_pot_label(ax, pot_str_)
        if corner:
            add_corner_label(ax, corner)

    def draw_ratio(ax, predv, dv, derr):
        with np.errstate(invalid="ignore", divide="ignore"):
            r    = np.where(predv > 0, dv / predv, np.nan)
            rerr = np.where(predv > 0, derr / predv, np.nan)
        inside = np.isfinite(r) & (r >= 0.0) & (r <= 2.0)
        ax.errorbar(bin_centers[inside], r[inside], yerr=rerr[inside],
                    fmt='o', markersize=4, markerfacecolor='black', markeredgecolor='black',
                    color='black', capsize=3, capthick=1.5, elinewidth=1.5)
        hi = np.isfinite(r) & (r > 2.0)
        if hi.any():
            ax.plot(bin_centers[hi], np.full(hi.sum(), 1.92), marker='^', linestyle='none',
                    markersize=5, color='red')
        ax.axhline(1.0, color='black', linestyle='--', linewidth=1.0, alpha=0.8)
        ax.set_ylim(0.0, 2.0)
        ax.set_ylabel(r"$\mathbf{Data/MC}$", fontsize=11)
        ax.set_xlabel(r"$\mathbf{" + xlabel.replace(" ", r"\ ") + "}$", fontsize=11)
        ax.grid(True, alpha=0.3)
        apply_tick_style(ax)

    pot_str = f"NuMI {ONBEAM_POT/1e19:.2f}" + r"$\times 10^{19}$ POT"
    corner_text = (r'$\mathbf{PPFX\ REWEIGHT\ APPLIED}$' + '\n' +
                   r'$\mathbf{CRT-PMT\ SPILL\ CUT\ APPLIED}$')

    draw_panel(tops[0], mc_h, rk_h, off_h, pred, dat_h, dat_err,
              r"$\mathbf{Events\ /\ Bin}$", pot_str, corner=corner_text)
    draw_ratio(bots[0], pred, dat_h, dat_err)

    draw_panel(tops[1], mc_a, rk_a, off_a, pred_a, dat_a, dat_a_err,
              r"$\mathbf{Fraction\ /\ Bin}$", "Area Normalized", corner=corner_text, show_counts=False)
    draw_ratio(bots[1], pred_a, dat_a, dat_a_err)

    os.makedirs(os.path.join(OUTPUT_DIR, "pdf"),  exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, "jpeg"), exist_ok=True)
    fig.savefig(os.path.join(OUTPUT_DIR, f"pdf/{output_suffix}.pdf"),  bbox_inches="tight", dpi=300)
    fig.savefig(os.path.join(OUTPUT_DIR, f"jpeg/{output_suffix}.jpeg"), bbox_inches="tight", dpi=300)
    plt.show()
    plt.close(fig)

    rock_note = "included" if include_rock else "excluded"
    print(f"\n--- {output_suffix}  [rock {rock_note}] ---")
    print(f"  data               : {n_data}")
    print(f"  prediction         : {pred.sum():.1f} +/- {np.sqrt((pred_e**2).sum()):.1f} (stat)")
    print(f"    nu+cosmic sim    : {mc_h.sum():.1f}")
    if include_rock:
        print(f"    rock muons       : {rk_h.sum():.1f}")
    print(f"    offbeam (cosmic) : {off_h.sum():.1f}")
    print(f"  data/MC            : {n_data/pred.sum():.4f} +/- "
          f"{_ratio_err(n_data, pred.sum(), np.sqrt((pred_e**2).sum())):.4f}")
    print(f"  folded into edge bins: underflow {u_tot}, overflow {o_tot}")

    return dict(n_data=n_data, pred=pred.sum())


# ============================================================================
# Flash time, flash total PE, and vertex x/y/z -- each with and without rock.
# ============================================================================
PLOT_SPECS = [
    dict(variable="reco_flash_time", xlabel="Flash Time [us]",
        bins=np.linspace(-5, 15, 51), name="preselection_flash_time",
        require_flash_match=False, valid_fn=None),
    dict(variable="reco_vertex_x", xlabel="Reconstructed Vertex X [cm]",
        bins=np.linspace(-400, 400, 51), name="preselection_vertex_x",
        require_flash_match=True, valid_fn=None),
    dict(variable="reco_vertex_y", xlabel="Reconstructed Vertex Y [cm]",
        bins=np.linspace(-200, 200, 51), name="preselection_vertex_y",
        require_flash_match=True, valid_fn=None),
    dict(variable="reco_vertex_z", xlabel="Reconstructed Vertex Z [cm]",
        bins=np.linspace(-1000, 1000, 51), name="preselection_vertex_z",
        require_flash_match=True, valid_fn=None),
]

for spec in PLOT_SPECS:
    for include_rock in (False, True):
        suffix = spec["name"] + ("_with_rock" if include_rock else "_no_rock")
        run_preselection_case(
            spec["variable"], spec["xlabel"], spec["bins"], suffix, include_rock,
            require_flash_match=spec["require_flash_match"], valid_fn=spec["valid_fn"])

print("\nDone.")



In [ ]:
# ============================================================================
# Rock-vs-cosmic angular separation study (advisor's suggestion).
#
# Physical picture: rock muons come from a beam neutrino interacting in the
# rock/dirt surrounding the cryostat (confirmed directly from the C++ source
# -- see evar::n_true_nu's docstring in event_variables.h), so they enter
# close to the NuMI beam direction, i.e. close to horizontal (along Z).
# Standalone cosmic-ray muons (the contamination we're trying to separate
# out, per Howard's Fig. 8 note on "simulated cosmic muons from before the
# beam spill providing the trigger") enter close to vertical, dominated by
# downward flux along -Y. This is the same variable Roy's technote calls
# Cos(theta_LTYD) (Sec. 4.1, Fig. 6): the cosine of the angle between the
# longest track and the vertical (Y) axis -- neutrino/beam-like tracks
# cluster near 0 (perpendicular to Y), cosmics cluster near -1 (downward).
#
# New branches (added to preselection.toml this session, selector =
# "longest_track" rather than the PID-based "leading_muon" used elsewhere
# in this notebook -- geometric, not PID-based, so it doesn't depend on a
# muon-score call to work on a rock event that may not PID as a clean muon):
#   reco_longest_track_start_dir_{x,y,z}, reco_longest_track_length
# Naming convention confirmed directly from framework.cc's construct():
# for type="reco_particle" with a selector, the branch is named
# "reco_" + selector + "_" + name.
#
# Cos(theta_LTYD) = start_dir_y already, PROVIDED start_dir is a unit
# vector; this cell computes it from the normalized (x,y,z) instead of
# assuming that, so a non-unit vector wouldn't silently bias the result.
#
# v3 change: the v2 run showed n_true_nu is an uninformative cross-check at
# this granularity (0 of 2,656,154 valid rows had n_true_nu==0 -- it's an
# event-level quantity broadcast onto preselection's per-INTERACTION rows,
# and can't discriminate between the ~14.5 reconstructed interactions that
# exist, on average, per rock-triggering event). true_category (already
# loaded, per-interaction) is used instead. v3 also finds the valley
# between the two lobes numerically rather than eyeballing it off the plot,
# since the v2 plot showed the valley is NOT at Roy's -0.7.
#
# Self-contained: re-imports and re-defines everything it needs.
# ============================================================================

import os
import numpy as np
import matplotlib.pyplot as plt
import uproot

# ------------------------------------------------------------------ config --
DATA_FILE = ("/Users/rvizarreta/Library/CloudStorage/GoogleDrive-rvizarreta14@gmail.com/"
             "My Drive/\U0001f3db PhD Repository/\U0001f680 Research/"
             "\U0001f916 Experiments&Projects/ICARUS/ICARUS_CC0pi_Selection/"
             "data/preselection.root")

OUTPUT_DIR = ("/Users/rvizarreta/Library/CloudStorage/GoogleDrive-rvizarreta14@gmail.com/"
              "My Drive/\U0001f3db PhD Repository/\U0001f680 Research/"
              "\U0001f916 Experiments&Projects/ICARUS/ICARUS_CC0pi_Selection/"
              "ICARUS-NuMI-CC0pi-Selection/spineplot/myAnalysis/"
              "1muNp0pi_Nge1_uncontained/preselection")

ROY_THRESHOLD = -0.7  # her cut: Cos(theta_LTYD) > -0.7 retains signal-like tracks

# ------------------------------------------------------------------ style ---
# Same conventions as the trigger-time/preselection-plots cells, so this
# figure matches the rest of the notebook rather than looking like a
# separate diagnostic script.
def apply_tick_style(ax):
    ax.tick_params(axis='both', which='major', labelsize=10, size=8, width=2, direction='in')
    ax.minorticks_on()
    ax.tick_params(axis='both', which='minor', size=4, width=1, direction='in')
    for lbl in ax.get_xticklabels() + ax.get_yticklabels():
        lbl.set_fontfamily('sans-serif')
    ax.yaxis.get_offset_text().set_fontfamily('sans-serif')
    ax.xaxis.get_offset_text().set_fontfamily('sans-serif')

def add_icarus_label(ax, subtitle="Work-in-Progress"):
    # Blue, not the "chocolate" brown used for the Data label in the other
    # cells -- this cell is Simulation-only (rock), so it gets its own color
    # rather than reusing the Data-labeled cells' convention.
    yr, xr = ax.get_ylim(), ax.get_xlim()
    ax.text(xr[0] + 0.01 * (xr[1] - xr[0]), yr[1] + 0.02 * (yr[1] - yr[0]),
            r'$\bf{ICARUS \cdot NuMI}$ Simulation' + '\n' + subtitle,
            fontsize=8, color="#003087", verticalalignment="bottom",
            fontfamily="sans-serif")

def add_pot_label(ax, pot_str):
    yr, xr = ax.get_ylim(), ax.get_xlim()
    ax.text(xr[1] - 0.01 * (xr[1] - xr[0]), yr[1] + 0.02 * (yr[1] - yr[0]), pot_str,
            fontsize=8, color="black", verticalalignment="bottom",
            horizontalalignment="right", fontfamily="sans-serif")

def add_corner_label(ax, text, y=0.85):
    ax.text(0.97, y, text, transform=ax.transAxes, fontsize=7, color='black',
            horizontalalignment='right', verticalalignment='top',
            fontfamily="sans-serif")

print(f"CELL VERSION: rock_angular_separation_v8_pot_label_2026-08-20")
print(f"Opening {DATA_FILE} ...")
f = uproot.open(DATA_FILE)

# Rock's own generated POT (this is a Simulation-only plot, so the relevant
# exposure to show is the rock production's POT, not onbeam data POT) --
# same histogram-first read pattern as read_exposure() in the trigger-time/
# preselection-plots cells.
if "events/rock/POT" in f:
    ROCK_POT = f["events/rock/POT"].values().sum()
    print(f"Rock POT (from events/rock/POT histogram): {ROCK_POT:.6e}")
else:
    key = "events/rock/preselection_exposure"
    if key not in f:
        raise KeyError("no exposure information found for rock "
                        "(looked for events/rock/POT and events/rock/preselection_exposure)")
    arrays = f[key].arrays(library="np")
    ROCK_POT = None
    for name, vals in arrays.items():
        if "pot" in name.lower():
            ROCK_POT = float(np.nansum(vals))
    if ROCK_POT is None:
        raise KeyError(f"no POT-like branch found in {key}: {list(arrays.keys())}")
    print(f"Rock POT (from {key}): {ROCK_POT:.6e}")

# ------------------------------------------------------------- load rock ---
# Pull the new geometric branches plus join keys and the existing true_category
# branch (already on preselection) for a first-pass sanity split.
rock_needed = ["Run", "Subrun", "Evt", "true_category"]
# Exact reco-level names, confirmed from framework.cc's construct(): for
# type="reco_particle" with a selector, the branch is "reco_" + selector +
# "_" + name. An endswith() filter alone is NOT safe here -- it matches
# both the "reco_" and "true_" versions of the same suffix (both_particle
# builds both), so we require the "reco_" prefix explicitly too.
rock_dir_candidates = {
    "start_dir_x": [c for c in f["events/rock/preselection"].keys()
                     if c.startswith("reco_") and c.endswith("longest_track_start_dir_x")],
    "start_dir_y": [c for c in f["events/rock/preselection"].keys()
                     if c.startswith("reco_") and c.endswith("longest_track_start_dir_y")],
    "start_dir_z": [c for c in f["events/rock/preselection"].keys()
                     if c.startswith("reco_") and c.endswith("longest_track_start_dir_z")],
    "length":      [c for c in f["events/rock/preselection"].keys()
                     if c.startswith("reco_") and c.endswith("longest_track_length")],
}
for key, cands in rock_dir_candidates.items():
    if len(cands) != 1:
        raise KeyError(f"expected exactly one column starting with 'reco_' and ending in "
                        f"'longest_track_{key}' on events/rock/preselection, found {cands}. "
                        f"Available columns: {list(f['events/rock/preselection'].keys())}")
DIR_COLS = {k: v[0] for k, v in rock_dir_candidates.items()}
print(f"Resolved longest_track direction columns: {DIR_COLS}")

rock = f["events/rock/preselection"].arrays(
    rock_needed + list(DIR_COLS.values()), library="pd")
print(f"rock/preselection: {len(rock)} rows")

# ------------------------------------- cross-check via true_category ------
# NOTE (v2 finding): n_true_nu is an event-level quantity -- broadcasting it
# onto preselection's per-INTERACTION rows can't discriminate between the
# ~14.5 reconstructed interactions that exist, on average, per rock-
# triggering event (2,785,031 preselection rows / 191,720 trigger_time
# events, from the v2 run). It came back 100% n_true_nu>0 with zero
# exceptions -- not a bug, just an uninformative cross-check at this
# granularity. true_category (already loaded above) is evaluated per
# INTERACTION, same granularity as cos_theta_y, so it's used instead:
# category 9 = "Cosmic" (cuts::neutrino fails for this specific
# interaction) is the relevant split.
print(f"\n  true_category value counts (rock/preselection, all rows):")
print(rock["true_category"].value_counts().sort_index().to_string())

# ------------------------------------------------------ compute cos(theta) --
vx = rock[DIR_COLS["start_dir_x"]].values
vy = rock[DIR_COLS["start_dir_y"]].values
vz = rock[DIR_COLS["start_dir_z"]].values
norm = np.sqrt(vx**2 + vy**2 + vz**2)

# Sentinel/invalid guard: rows with no valid longest-track direction (e.g. no
# reconstructed particle at all in that interaction) show up as norm <= 0 or
# non-finite -- exclude them explicitly rather than silently propagating NaN
# into the histogram.
valid = np.isfinite(norm) & (norm > 0)
n_invalid = int((~valid).sum())
print(f"\n  excluded as invalid longest-track direction (norm <= 0 or non-finite): "
      f"{n_invalid} / {len(rock)} rows ({n_invalid/len(rock):.1%})")

cos_theta_y = np.full(len(rock), np.nan)
cos_theta_y[valid] = vy[valid] / norm[valid]

true_cat = rock["true_category"].values
is_cosmic_cat = valid & np.isfinite(true_cat) & (true_cat == 9)
is_nu_cat     = valid & np.isfinite(true_cat) & (true_cat != 9)

# ------------------------------------------------------- valley-finding ----
# Numeric local minimum between the two lobes, from the binned counts
# themselves (not eyeballed off the plot). Restrict the search to the
# interior of the range so an edge bin can't spuriously "win".
BINS = np.linspace(-1.0, 1.0, 51)
bin_centers = 0.5 * (BINS[:-1] + BINS[1:])
counts_all, _ = np.histogram(cos_theta_y[valid], bins=BINS)
search_lo, search_hi = 15, 35  # bin_centers ~ -0.4 to +0.4, away from both peaks
interior = counts_all[search_lo:search_hi]
valley_bin = search_lo + int(np.argmin(interior))
VALLEY = bin_centers[valley_bin]
print(f"\n  numeric valley (local min of binned counts, searched in "
      f"[{bin_centers[search_lo]:.2f}, {bin_centers[search_hi]:.2f}]): "
      f"Cos(theta_LTYD) = {VALLEY:.3f} (bin count = {counts_all[valley_bin]})")

# --------------------------------------------------------------- plotting ---
# Single panel, full longest-track angular distribution, ICARUS-style
# cosmetics (matching the trigger-time/preselection-plots cells) -- no
# true_category split panel, per instruction.
fig, ax = plt.subplots(figsize=(5, 4))
ax.hist(cos_theta_y[valid], bins=BINS, histtype="stepfilled",
        color="seagreen", alpha=0.7, label="Rock sample")
ax.axvline(VALLEY, color="k", linestyle=":", linewidth=1.5,
           label=f"Numeric valley ({VALLEY:.2f})")
ax.set_xlabel(r"Cos($\theta_{LTYD}$)  [= longest-track start_dir$_y$]")
ax.set_ylabel("Interactions")
ax.set_xlim(-1.0, 1.0)
ax.set_ylim(bottom=0)
ax.legend(fontsize=9, loc="upper right")
apply_tick_style(ax)
add_icarus_label(ax)
add_pot_label(ax, f"NuMI {ROCK_POT/1e19:.2f}" + r"$\times 10^{19}$ POT")
fig.tight_layout()

os.makedirs(os.path.join(OUTPUT_DIR, "pdf"),  exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "jpeg"), exist_ok=True)
fig.savefig(os.path.join(OUTPUT_DIR, "pdf/rock_angular_separation.pdf"),
            bbox_inches="tight", dpi=300)
fig.savefig(os.path.join(OUTPUT_DIR, "jpeg/rock_angular_separation.jpeg"),
            bbox_inches="tight", dpi=300)
plt.show()
plt.close(fig)

# ------------------------------------------------------------- summary ------
print(f"\n--- rock, Cos(theta_LTYD) summary ---")
print(f"  total rows                        : {len(rock)}")
print(f"  valid (finite, non-degenerate dir) : {valid.sum()} ({valid.sum()/len(rock):.1%})")

below_roy = valid & (cos_theta_y <= ROY_THRESHOLD)
above_roy = valid & (cos_theta_y >  ROY_THRESHOLD)
print(f"\n  at Roy's threshold ({ROY_THRESHOLD}), among valid rows:")
print(f"    cos_theta_y <= {ROY_THRESHOLD} (would be CUT)  : "
      f"{below_roy.sum()} ({below_roy.sum()/valid.sum():.1%})")
print(f"    cos_theta_y >  {ROY_THRESHOLD} (kept)           : "
      f"{above_roy.sum()} ({above_roy.sum()/valid.sum():.1%})")

below_val = valid & (cos_theta_y <= VALLEY)
above_val = valid & (cos_theta_y >  VALLEY)
print(f"\n  at the numeric valley ({VALLEY:.3f}), among valid rows:")
print(f"    cos_theta_y <= {VALLEY:.3f} (would be CUT)  : "
      f"{below_val.sum()} ({below_val.sum()/valid.sum():.1%})")
print(f"    cos_theta_y >  {VALLEY:.3f} (kept)           : "
      f"{above_val.sum()} ({above_val.sum()/valid.sum():.1%})")

for label, mask in (("true_category != 9", is_nu_cat), ("true_category == 9 (Cosmic)", is_cosmic_cat)):
    if mask.sum() == 0:
        continue
    fr_roy = (mask & below_roy).sum() / mask.sum()
    fr_val = (mask & below_val).sum() / mask.sum()
    print(f"  within '{label}' ({mask.sum()} rows): "
          f"{fr_roy:.1%} fall below Roy's threshold, {fr_val:.1%} fall below the numeric valley")

print("\nDone. Compare the numeric-valley split against true_category above -- if "
      "true_category==9 rows are concentrated below the valley and true_category!=9 "
      "rows above it, the valley is a well-motivated cut to apply to the rock sample "
      "before the trigger-time plot.")
